# M01-01 — Sesión Spark y primer DataFrame

[← Anterior](01-teoria.ipynb) · [Siguiente →](../M02-ingesta-preparacion/01-teoria.ipynb)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

Dejar una SparkSession viva y materializar 5 pedidos en memoria. Distingues un `filter` de un `count`. Aún no lees `data/raw/`.

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M01-01-sesion-spark.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras.
2. **Código** — el de la celda de código del paso. Lo ejecutas (`Shift+Enter`), miras la salida y, si no cuadra, lo mejoras.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


### Paso 1 — Arranque (siempre el primero)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Celda 0: localizo el repo y las rutas. Sin esto el resto no arranca.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `RAW` existe True. `ROOT` es la carpeta del curso.

**Por qué este paso.** El notebook vive en `trabajo/`; las rutas se resuelven desde el repo, no desde `cwd`.

**Si no sale.** Copia la celda entera. No escribas `Path('data/raw')`.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


### Paso 2 — Comprueba Java y PySpark

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Antes de crear la sesión miro versiones. Sin JRE 17 Spark no arranca.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** PySpark `3.5.x` y una línea `openjdk version "17…"` (o Microsoft JDK 17).

**Por qué este paso.** Detectas el fallo de entorno *antes* de pelearte con un DataFrame.

**Si no sale.** Si Java no es 17: Codespace limpio o `java -version` en local. No improvises otro JDK.


In [ ]:
import pyspark, shutil, subprocess

print("pyspark", pyspark.__version__)
print(subprocess.check_output(["java", "-version"], text=True, stderr=subprocess.STDOUT).splitlines()[0])
print("java:", shutil.which("java"))


### Paso 3 — Crea (o reusa) la sesión

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Pido una SparkSession local[*]. getOrCreate evita un segundo contexto en el puerto 4040.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Ves un objeto SparkSession. Vuelve a ejecutar la misma celda: es **la misma** sesión, no otra.

**Por qué este paso.** `get_spark` ya pone master `local[*]`, UI 4040 y TZ UTC.

**Si no sale.** Puerto ocupado: `spark.stop()` y otra vez `get_spark()`.


In [ ]:
spark = get_spark("novashop-m01")
spark


### Paso 4 — Cinco pedidos en memoria

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

createDataFrame es la forma más pequeña de ver schema + tabla sin ficheros.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Schema: `order_id`/`customer_id`/`status` string, `amount` double. Tabla de 5 filas (O90001…O90005).

**Por qué este paso.** Materializas algo visible. `printSchema` y `show` **sí** son acciones.


In [ ]:
from pyspark.sql import Row

pedidos = [
    Row(order_id="O90001", customer_id="C0001", status="paid", amount=49.90),
    Row(order_id="O90002", customer_id="C0002", status="paid", amount=12.50),
    Row(order_id="O90003", customer_id="C0003", status="cancelled", amount=80.00),
    Row(order_id="O90004", customer_id="C0001", status="paid", amount=23.10),
    Row(order_id="O90005", customer_id="C0004", status="pending", amount=5.00),
]
df = spark.createDataFrame(pedidos)
df.printSchema()
df.show()


### Paso 5 — Filter no cuenta; count sí

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

filter solo alarga el plan. count obliga a ejecutarlo. Quiero 3 paid.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `paid count = 3`.

**Por qué este paso.** Si crees que `filter` ya filtró y “no ves nada”, te falta una acción.

Opcional, en otra celda (con su Markdown): `paid.explain("formatted")`. No hace falta entender cada línea.


In [ ]:
paid = df.filter(df.status == "paid")
print("después del filter, Spark aún no ha contado nada")
print("paid count =", paid.count())


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

- `spark.version` es 3.5.x.
- Sobre las 5 filas, `status == "paid"` da **3**.
- Tu notebook tiene Markdown **antes** de cada código.
- **Run All** funciona de arriba abajo.


## Mejora — Canal en dos columnas

En *tu* notebook: Markdown que explique el `withColumn` + código que añada `channel` con todos `"web"` y muestre solo `order_id` y `channel`. Ejecuta. Deben ser 5 filas y no debe salir `amount`.

Si te atasca, el código está en la celda siguiente.


In [ ]:
from pyspark.sql.functions import lit

df.withColumn("channel", lit("web")).select("order_id", "channel").show()


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| `Java gateway process exited` | No hay JDK 17 | Codespace limpio; `java -version` = 17 |
| Puerto 4040 ocupado | Segunda SparkSession | `spark.stop()` y `get_spark()` |
| `filter` y “no veo nada” | No lanzaste acción | Encadena `.show()` o `.count()` |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [M02 — teoría](../M02-ingesta-preparacion/01-teoria.ipynb).
